[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AltamarMx/anomalias-2026-2/blob/main/notebooks/015_autocorrelacion_teoria.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
rng = np.random.default_rng(42)
import ipywidgets as widgets
from ipywidgets import interact, interactive_output
from IPython.display import display, Markdown

# Semana 8A — Autocorrelación y Estacionariedad (Teoría)

**Objetivo:** Entender la autocorrelación como herramienta diagnóstica
para series temporales, distinguir ACF de PACF, y desarrollar intuición
sobre estacionariedad.

**Temas:**

1. Concepto de autocorrelación y ACF
2. Autocorrelación parcial (PACF)
3. Estacionariedad intuitiva
4. Diferenciación para lograr estacionariedad

---
## 1. Autocorrelación (ACF)

La **autocorrelación en lag $k$** mide cuánto se parece una serie
a sí misma desplazada $k$ pasos:

$$\text{ACF}(k) = \text{Corr}(y_t,\; y_{t+k})$$

- ACF(0) = 1 siempre (la serie es perfectamente correlacionada consigo misma).
- ACF(k) → 0 para series aleatorias (ruido blanco).
- Si ACF(k) decae lentamente → la serie tiene *memoria* (persistencia).

La forma del ACF **delata el proceso generador** de la serie.

### Serie AR(1): $y_t = \phi \cdot y_{t-1} + \varepsilon_t$

In [ ]:
phi_slider = widgets.FloatSlider(min=-0.95, max=0.95, step=0.05, value=0.9, description="φ (coeficiente AR(1))",
)

def _update(phi_slider):
    _phi = phi_slider
    _n = 500
    _y = np.zeros(_n)
    for t in range(1, _n):
        _y[t] = _phi * _y[t - 1] + rng.normal(0, 1)

    _acf_vals = acf(_y, nlags=30, fft=True)
    _pacf_vals = pacf(_y, nlags=30)

    fig1, axes1 = plt.subplots(1, 3, figsize=(14, 4))

    # Serie
    axes1[0].plot(_y, "steelblue", lw=0.6)
    axes1[0].set_xlabel("t")
    axes1[0].set_ylabel("y")
    axes1[0].set_title(f"Serie AR(1)  (φ = {_phi})")

    # ACF
    axes1[1].bar(range(len(_acf_vals)), _acf_vals, color="steelblue",
                 edgecolor="white", width=0.8)
    axes1[1].axhline(0, color="black", lw=0.5)
    _ci = 1.96 / np.sqrt(_n)
    axes1[1].axhline(_ci, color="crimson", ls="--", lw=1, alpha=0.6)
    axes1[1].axhline(-_ci, color="crimson", ls="--", lw=1, alpha=0.6)
    axes1[1].set_xlabel("Lag")
    axes1[1].set_title("ACF")

    # PACF
    axes1[2].bar(range(len(_pacf_vals)), _pacf_vals, color="darkorange",
                 edgecolor="white", width=0.8)
    axes1[2].axhline(0, color="black", lw=0.5)
    axes1[2].axhline(_ci, color="crimson", ls="--", lw=1, alpha=0.6)
    axes1[2].axhline(-_ci, color="crimson", ls="--", lw=1, alpha=0.6)
    axes1[2].set_xlabel("Lag")
    axes1[2].set_title("PACF")

    plt.tight_layout()
    plt.show()

out = widgets.interactive_output(_update, {'phi_slider': phi_slider})
display(widgets.VBox([phi_slider, out]))

> **Experimenta con φ:**
> - **φ = 0.9:** ACF decae lentamente (mucha memoria); PACF corta
>   abruptamente en lag 1.
> - **φ = 0:** Ruido blanco — ACF ≈ 0 para todos los lags.
> - **φ = −0.5:** ACF alterna entre + y − (oscilaciones); PACF
>   sigue cortando en lag 1.
>
> **Firma del AR(1):** ACF decae exponencialmente, PACF corta en lag 1.

### Serie MA(1): $y_t = \varepsilon_t + \theta \cdot \varepsilon_{t-1}$

In [ ]:
theta_slider = widgets.FloatSlider(min=-0.95, max=0.95, step=0.05, value=0.7, description="θ (coeficiente MA(1))",
)

def _update(theta_slider):
    _theta = theta_slider
    _n = 500
    _eps = rng.normal(0, 1, _n)
    _y = np.zeros(_n)
    for t in range(1, _n):
        _y[t] = _eps[t] + _theta * _eps[t - 1]

    _acf_vals = acf(_y, nlags=30, fft=True)
    _pacf_vals = pacf(_y, nlags=30)

    fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4))

    axes2[0].plot(_y, "seagreen", lw=0.6)
    axes2[0].set_xlabel("t")
    axes2[0].set_ylabel("y")
    axes2[0].set_title(f"Serie MA(1)  (θ = {_theta})")

    _ci = 1.96 / np.sqrt(_n)

    axes2[1].bar(range(len(_acf_vals)), _acf_vals, color="seagreen",
                 edgecolor="white", width=0.8)
    axes2[1].axhline(0, color="black", lw=0.5)
    axes2[1].axhline(_ci, color="crimson", ls="--", lw=1, alpha=0.6)
    axes2[1].axhline(-_ci, color="crimson", ls="--", lw=1, alpha=0.6)
    axes2[1].set_xlabel("Lag")
    axes2[1].set_title("ACF")

    axes2[2].bar(range(len(_pacf_vals)), _pacf_vals, color="darkorange",
                 edgecolor="white", width=0.8)
    axes2[2].axhline(0, color="black", lw=0.5)
    axes2[2].axhline(_ci, color="crimson", ls="--", lw=1, alpha=0.6)
    axes2[2].axhline(-_ci, color="crimson", ls="--", lw=1, alpha=0.6)
    axes2[2].set_xlabel("Lag")
    axes2[2].set_title("PACF")

    plt.tight_layout()
    plt.show()

out = widgets.interactive_output(_update, {'theta_slider': theta_slider})
display(widgets.VBox([theta_slider, out]))

> **Firma del MA(1):** ACF tiene un solo pico en lag 1 y luego corta
> a cero; PACF decae gradualmente.
>
> Esto es exactamente lo contrario del AR(1).

---
## 2. ACF vs PACF — Tabla resumen

La PACF en lag $k$ mide la correlación entre $y_t$ e $y_{t+k}$
**removiendo** el efecto de los lags intermedios ($y_{t+1}, ..., y_{t+k-1}$).

| Proceso | ACF | PACF |
|:---|:---|:---|
| **AR(p)** | Decae gradualmente | Corta abruptamente en lag $p$ |
| **MA(q)** | Corta abruptamente en lag $q$ | Decae gradualmente |
| **ARMA(p,q)** | Decae gradualmente | Decae gradualmente |
| **Ruido blanco** | ≈ 0 para todo lag > 0 | ≈ 0 para todo lag > 0 |

Esta tabla es la herramienta clave para **identificar el tipo de modelo**
que mejor describe una serie temporal.  La usaremos en la semana 12
para elegir los órdenes de ARIMA.

---
## 3. Estacionariedad intuitiva

Una serie es **estacionaria** si sus propiedades estadísticas
(media, varianza, autocorrelación) **no cambian con el tiempo**.

- **Estacionaria:** ruido blanco, AR(1) estable
- **No estacionaria en media:** tiene tendencia (la media cambia)
- **No estacionaria en varianza:** la variabilidad cambia con el tiempo

In [ ]:
_n = 500

# (a) Estacionaria — AR(1)
_y_est = np.zeros(_n)
for t in range(1, _n):
    _y_est[t] = 0.7 * _y_est[t - 1] + rng.normal(0, 1)

# (b) No estacionaria en media — tendencia lineal
_y_tend = 0.05 * np.arange(_n) + rng.normal(0, 1, _n)

# (c) No estacionaria en varianza — volatilidad creciente
_y_var = rng.normal(0, 1, _n) * np.linspace(0.5, 5, _n)

_series = [
    (_y_est, "Estacionaria (AR(1))", "steelblue"),
    (_y_tend, "No estacionaria — media", "crimson"),
    (_y_var, "No estacionaria — varianza", "darkorange"),
]

fig3, axes3 = plt.subplots(3, 2, figsize=(13, 9))

_ventana = 50
for row, (y, titulo, color) in enumerate(_series):
    # Serie
    axes3[row, 0].plot(y, color=color, lw=0.6)
    axes3[row, 0].set_title(titulo)
    axes3[row, 0].set_ylabel("y")

    # Media y DE por ventanas
    _medias = []
    _des = []
    _centros = []
    for start in range(0, _n - _ventana, _ventana // 2):
        _bloque = y[start : start + _ventana]
        _medias.append(_bloque.mean())
        _des.append(_bloque.std())
        _centros.append(start + _ventana // 2)

    ax_r = axes3[row, 1]
    ax_r.plot(_centros, _medias, "o-", color=color, markersize=4,
              lw=1.5, label="Media")
    ax_twin = ax_r.twinx()
    ax_twin.plot(_centros, _des, "s--", color="gray", markersize=4,
                 lw=1.5, label="DE", alpha=0.7)
    ax_r.set_ylabel("Media", color=color)
    ax_twin.set_ylabel("DE", color="gray")
    ax_r.set_title("Media y DE por ventanas")
    if row == 2:
        ax_r.set_xlabel("t")

plt.tight_layout()
plt.show()

> **Observa en los paneles derechos:**
> - **Estacionaria:** Media y DE oscilan alrededor de valores constantes.
> - **No estacionaria en media:** La media crece linealmente;
>   la DE es constante.
> - **No estacionaria en varianza:** La media puede ser estable
>   pero la DE crece con el tiempo.
>
> **¿Por qué importa?** Muchos métodos estadísticos (ICs, pruebas t,
> ARIMA) asumen estacionariedad.  Si la serie no lo es, los resultados
> pueden ser espurios.

---
## 4. Diferenciación para lograr estacionariedad

Si la serie tiene tendencia, la **diferenciación** la remueve:

$$\Delta y_t = y_t - y_{t-1}$$

- Si una diferenciación basta → la serie era **integrada de orden 1** (I(1)).
- Si necesitamos diferenciar dos veces → I(2).
- La "I" en ARIMA viene de aquí: ARIMA(p, **d**, q), donde $d$
  es el orden de diferenciación.

In [ ]:
_n = 500
# Random walk (no estacionario): y_t = y_{t-1} + ε
_y_rw = np.cumsum(rng.normal(0, 1, _n))

# Diferenciada
_dy = np.diff(_y_rw)

_acf_orig = acf(_y_rw, nlags=30, fft=True)
_acf_diff = acf(_dy, nlags=30, fft=True)

fig4, axes4 = plt.subplots(2, 2, figsize=(12, 7))

# Original
axes4[0, 0].plot(_y_rw, "crimson", lw=0.8)
axes4[0, 0].set_title("Random walk (no estacionario)")
axes4[0, 0].set_ylabel("y")

axes4[0, 1].bar(range(len(_acf_orig)), _acf_orig, color="crimson",
                edgecolor="white", width=0.8)
axes4[0, 1].axhline(0, color="black", lw=0.5)
axes4[0, 1].set_title("ACF del random walk")

# Diferenciada
axes4[1, 0].plot(_dy, "seagreen", lw=0.6)
axes4[1, 0].set_title("Δy (diferenciada) = ruido blanco")
axes4[1, 0].set_ylabel("Δy")
axes4[1, 0].set_xlabel("t")

axes4[1, 1].bar(range(len(_acf_diff)), _acf_diff, color="seagreen",
                edgecolor="white", width=0.8)
axes4[1, 1].axhline(0, color="black", lw=0.5)
_ci = 1.96 / np.sqrt(len(_dy))
axes4[1, 1].axhline(_ci, color="crimson", ls="--", lw=1, alpha=0.6)
axes4[1, 1].axhline(-_ci, color="crimson", ls="--", lw=1, alpha=0.6)
axes4[1, 1].set_title("ACF de Δy")
axes4[1, 1].set_xlabel("Lag")

plt.tight_layout()
plt.show()

> **Resultado:**
> - El random walk tiene ACF que decae muy lentamente (señal de
>   no estacionariedad).
> - Después de diferenciar, la serie se convierte en ruido blanco
>   (ACF ≈ 0 para todos los lags) → estacionaria.
> - En la sesión de laboratorio verificaremos que `tdb` diaria
>   necesita diferenciación para ser estacionaria.

---
## Resumen

| Concepto | Idea clave |
|:---|:---|
| **ACF(k)** | Correlación de la serie consigo misma desplazada $k$ pasos |
| **PACF(k)** | Correlación en lag $k$ removiendo el efecto de lags intermedios |
| **AR(p)** | ACF decae, PACF corta en lag $p$ |
| **MA(q)** | ACF corta en lag $q$, PACF decae |
| **Estacionariedad** | Media, varianza y ACF constantes en el tiempo |
| **Diferenciación** | $\Delta y_t = y_t - y_{t-1}$ remueve tendencias; la "I" de ARIMA |

**Próxima sesión (8B):** ACF/PACF de datos reales, detección de gaps
y control de calidad del dataset ClimaLab.